# Proyecto EDO Y NUMERICA

In [ ]:
#!/usr/bin/env python3
"""
Generate isoclines and slope field for Part A (Newton cooling):
    dT/dt = -k (T - Ta),
with Ta = 70 F. Saves image to Figures/isoclinas_partA.png
Also computes k from observations and estimates time of death.
"""
import math
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Parameters
Ta = 70.0
k = math.log(2.0)  # theoretical value from the data provided

a = 0.0
b = 6.0  # hours to display
Tmin = 55
Tmax = 105

# Grid for slope field (reduced density and normalized arrows to avoid overlap)
t_vals = np.linspace(a, b, 16)  # fewer columns (less horizontal crowding)
T_vals = np.linspace(Tmin, Tmax, 18)  # reasonable vertical spacing
Tg, tg = np.meshgrid(T_vals, t_vals)
# compute dT/dt at each T
dTdt = -k * (Tg - Ta)
# create base vectors and normalize them so arrows have uniform visual length
dt_step = (b - a) / (len(t_vals) - 1)
U_raw = dt_step * np.ones_like(dTdt)
V_raw = dTdt.copy()
# normalize and scale to a controllable arrow length
arrow_len = dt_step * 0.6
mag = np.hypot(U_raw, V_raw)
mag[mag == 0] = 1.0
U = U_raw / mag * arrow_len
V = V_raw / mag * arrow_len

plt.figure(figsize=(8,6))
plt.quiver(tg, Tg, U, V, angles='xy', scale_units='xy', scale=1, color='tab:gray', width=0.0025, alpha=0.85)

# Plot isoclines: choose a set of slopes (degF per hour) and compute T = Ta - s/k
slopes = [-4.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0]
for i, s in enumerate(slopes):
    T_iso = Ta - s / k
    if Tmin - 5 <= T_iso <= Tmax + 5:
        plt.hlines(T_iso, a, b, linestyles='--', colors='C1', alpha=0.8)
        # stagger label vertically to avoid overlapping labels near Ta
        y_offset = ((-1) ** i) * 0.9  # alternate up/down about 0.9 °F
        plt.text(b + 0.05, T_iso + y_offset, f's={s:.1f}', va='center', fontsize=8)

# Sample solution curves for representative initial temps
t_plot = np.linspace(a, b, 400)
for T0, lab in [(98.6, 'T0=98.6 F'), (85.0, 'T0=85 F'), (65.0, 'T0=65 F')]:
    Tt = Ta + (T0 - Ta) * np.exp(-k * t_plot)
    plt.plot(t_plot, Tt, label=lab)

# Emphasize ambient temperature Ta = 70 F
plt.axhline(Ta, color='k', linewidth=2, alpha=0.9)
plt.text(b + 0.05, Ta, f'Ta = {Ta:.0f} °F', va='center', fontsize=9, fontweight='bold')

# Ensure 70 is visible on y-ticks
yticks = list(plt.yticks()[0])
if 70.0 not in yticks:
    yticks.append(70.0)
plt.yticks(sorted(yticks))
plt.grid(alpha=0.3)

plt.xlim(a, b + 0.8)
plt.ylim(Tmin, Tmax)
plt.xlabel('t (h)')
plt.ylabel('T (°F)')
plt.title('Campo de pendientes e isoclinas — Parte A (Ley de Newton)')
plt.legend(loc='lower right')
plt.tight_layout()
outfile = 'Figures/isoclinas_partA.png'
outdir = os.path.dirname(outfile)
if outdir:
    os.makedirs(outdir, exist_ok=True)
plt.savefig(outfile, dpi=200)
print('Saved', outfile)

# --- Additional computations: estimate k from the observed temps and time of death ---
# Observations: at 12:00 (t=0) T=80 F; at 13:00 (t=1) T=75 F; ambient Ta = 70 F
T_noon = 80.0
T_1pm = 75.0
# compute k from T(0)=80 and T(1)=75: (T1-Ta)/(T0-Ta) = e^{-k}
k_est = -math.log((T_1pm - Ta) / (T_noon - Ta))
print(f'k (from observations) = {k_est:.5f} 1/h (theoretical ln(2)={math.log(2):.5f})')

# Estimate how long before noon the death occurred, assuming T_death = 98.6 F at time of death
T_death = 98.6
# Relationship: T_noon = Ta + (T_death - Ta) * exp(-k_est * s), where s is hours between death and noon
s = -1.0 / k_est * math.log((T_noon - Ta) / (T_death - Ta))
# Convert to h,m,s
hours = int(s)
minutes = int((s - hours) * 60)
seconds = int(round(((s - hours) * 60 - minutes) * 60))
time_of_death = (datetime(2000,1,1,12,0,0) - timedelta(hours=s)).time()
print(f'Estimated time since death at noon: {s:.3f} h = {hours} h {minutes} min {seconds} s before noon.')
print(f'Estimated time of death ≈ {time_of_death} (about {s*60:.1f} minutes before noon => around {time_of_death.strftime("%I:%M %p")})')

print("Esto funciona")

Saved Figures/isoclinas_partA.png
k (from observations) = 0.69315 1/h (theoretical ln(2)=0.69315)
Estimated time since death at noon: 1.516 h = 1 h 30 min 58 s before noon.
Estimated time of death ≈ 10:29:02.345471 (about 91.0 minutes before noon => around 10:29 AM)
Esto funciona
